# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TobyRathmell123/ML-Internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
# ============================================================
# 0. SETUP — connect to the Hugging Face warehouse
# ============================================================
%pip -q install duckdb huggingface_hub

import os, getpass
import pandas as pd
import numpy as np
import duckdb

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

print("Connected. Row counts (touches metadata only, near-free):")
for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"  {name:12} {n:>12,} rows")

# ============================================================
# 1. BUILD PREV30 / LAST30 WINDOWS IN SQL (the heavy cell — a
#    few minutes; if it times out or you hit HTTP 429, swap
#    TABLES['fact_daily'] for TABLES['fact_daily_sample'] while
#    testing, then switch back for your real final run)
# ============================================================
MIN_IMPRESSIONS = 50  # volume floor — below this, CTR swings are noise, not signal

features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,
            SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
            SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
            SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
            SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_prev30,
            AVG(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS avg_position_prev30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= {MIN_IMPRESSIONS} AND imp_last30 >= {MIN_IMPRESSIONS}
    )
    SELECT * FROM windowed
""").df()

print(f"\n{len(features):,} content items with enough volume in both windows")

# ============================================================
# 2. CTR COLUMNS + LABEL (absolute point-gain threshold —
#    this is the fix from last time, keeping the earlier
#    percentage-multiplier artifact out)
# ============================================================
features["ctr_prev30"] = features["clk_prev30"] / features["imp_prev30"]
features["ctr_last30"] = features["clk_last30"] / features["imp_last30"]

ABSOLUTE_GROWTH_THRESHOLD = 0.003  # CTR must rise by at least 0.3 percentage points
features["is_ctr_growing_label"] = (
    (features["ctr_last30"] - features["ctr_prev30"]) >= ABSOLUTE_GROWTH_THRESHOLD
).astype(int)

print("Base rate (share labeled 'growing'):", round(features["is_ctr_growing_label"].mean(), 3))

# ============================================================
# 3. JOIN CONTENT METADATA
# ============================================================
content_meta = con.sql(f"""
    SELECT
        content_hash_id,
        content_type,
        main_intent,
        competition_level,
        word_count,
        char_count,
        search_volume,
        competition,
        cpc,
        backlinks
    FROM {TABLES['dim_content']}
""").df()

data = features.merge(content_meta, on="content_hash_id", how="left")
print(f"\nJoined content metadata. {len(data):,} rows.")

# ============================================================
# 4. LEAKAGE GUARD
# ============================================================
leaky_columns = ["clk_last30", "imp_last30", "ctr_last30"]

safe_features_numeric = [
    "imp_prev30", "clk_prev30", "ctr_prev30", "avg_position_prev30",
    "word_count", "char_count", "search_volume", "competition", "cpc", "backlinks",
]
safe_features_categorical = [
    "content_type", "main_intent", "competition_level",
]

overlap = set(leaky_columns) & set(safe_features_numeric + safe_features_categorical)
assert not overlap, f"Leaky column found in features: {overlap}"
print("No leakage found in feature list.")

# ============================================================
# 5. DIAGNOSTIC — check the real scale of ctr_prev30 before
#    picking a baseline threshold. Last run's baseline scored
#    0.000, which could mean either (a) a real finding — pages
#    already ranking well have no CTR headroom left to grow —
#    or (b) the old 0.5 threshold was borrowed from the starter
#    CSV's scale and doesn't mean anything on this data. This
#    print tells us which.
# ============================================================
print("\nctr_prev30 distribution:")
print(data["ctr_prev30"].describe())

median_ctr = data["ctr_prev30"].median()
print(f"\nUsing median ctr_prev30 ({median_ctr:.4f}) as the 'weak CTR' cutoff for the baseline.")

# ============================================================
# 6. BASELINE RULE — threshold now scaled to this dataset,
#    not borrowed from the starter CSV
# ============================================================
def ctr_growth_baseline_score(row, ctr_weak_threshold):
    if row["avg_position_prev30"] <= 0 or pd.isna(row["avg_position_prev30"]) or row["ctr_prev30"] <= 0:
        return 0.0
    position_opportunity = 1 - min(row["avg_position_prev30"], 50) / 50
    ctr_weakness = max(0, ctr_weak_threshold - row["ctr_prev30"])
    return position_opportunity * ctr_weakness * row["imp_prev30"]

data["baseline_score"] = data.apply(lambda r: ctr_growth_baseline_score(r, median_ctr), axis=1)

# Quick look at what the baseline actually picks, so you can eyeball whether
# it makes sense before trusting the precision number below.
print("\nBaseline's top 10 picks:")
print(
    data.sort_values("baseline_score", ascending=False)
    .head(10)[["avg_position_prev30", "ctr_prev30", "ctr_last30", "imp_prev30", "is_ctr_growing_label"]]
)

# ============================================================
# 7. HONEST SPLIT — grouped by client
# ============================================================
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(data, groups=data["client_hash_id"]))

train_df = data.iloc[train_idx]
test_df = data.iloc[test_idx]

overlap_clients = set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])
print(f"\nTrain: {len(train_df):,} | Test: {len(test_df):,} | Clients in both: {len(overlap_clients)} (should be 0)")

# ============================================================
# 8. BUILD FEATURES + TRAIN THE RANDOM FOREST
# ============================================================
from sklearn.ensemble import RandomForestClassifier

def build_features(frame):
    numeric = frame[safe_features_numeric].apply(pd.to_numeric, errors="coerce").fillna(0)
    categorical = pd.get_dummies(frame[safe_features_categorical].fillna("unknown").astype(str))
    return pd.concat([numeric.reset_index(drop=True), categorical.reset_index(drop=True)], axis=1)

X_train = build_features(train_df)
X_test = build_features(test_df)
X_train, X_test = X_train.align(X_test, join="outer", axis=1, fill_value=0)

y_train = train_df["is_ctr_growing_label"].values
y_test = test_df["is_ctr_growing_label"].values

model = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=25,
    class_weight="balanced_subsample", random_state=42, n_jobs=-1,
)
model.fit(X_train, y_train)
probabilities = model.predict_proba(X_test)[:, 1]

# Feature importances — worth a look each time, to catch any single
# feature suspiciously dominating (a possible leakage sign).
importances = pd.Series(model.feature_importances_, index=X_train.columns)
print("\nFeature importances:")
print(importances.sort_values(ascending=False))

# ============================================================
# 9. COMPARE MODEL VS BASELINE — same split, same metric
# ============================================================
def precision_at_k(labels, scores, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

baseline_test_scores = test_df["baseline_score"].values

print("\n--- Results ---")
for k in (20, 50):
    base = precision_at_k(y_test, baseline_test_scores, k)
    rf = precision_at_k(y_test, probabilities, k)
    print(f"Precision@{k}:  baseline {base:.3f}   |   random forest {rf:.3f}")

print("Base rate in test set:", round(y_test.mean(), 3))

Paste your Hugging Face READ token (hf_...): ··········
Connected. Row counts (touches metadata only, near-free):
  dim_clients           104 rows
  dim_content       519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  fact_daily     78,835,655 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


107,324 content items with enough volume in both windows
Base rate (share labeled 'growing'): 0.188


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Joined content metadata. 107,324 rows.
No leakage found in feature list.

ctr_prev30 distribution:
count    107324.000000
mean          0.003527
std           0.005408
min           0.000000
25%           0.000000
50%           0.001809
75%           0.004890
max           0.151261
Name: ctr_prev30, dtype: float64

Using median ctr_prev30 (0.0018) as the 'weak CTR' cutoff for the baseline.

Baseline's top 10 picks:
       avg_position_prev30  ctr_prev30  ctr_last30  imp_prev30  \
60886             5.754623    0.000187    0.000136    230512.0   
41526             2.543075    0.000243    0.002000    213610.0   
14170            10.091173    0.000011    0.000590    178306.0   
11017            14.020197    0.000162    0.000522    191252.0   
88867             7.787938    0.000227    0.000187    140919.0   
42867             3.065436    0.001254    0.001364    349963.0   
11153             7.532791    0.000036    0.000448    112236.0   
64953            10.591592    0.000242    0.000436  

## 1. Question

*The research question and the decision it supports.*

In [ ]:
"""
**Research question:** Can we predict, from a page's search performance in one 30-day
window, whether its click-through rate (CTR) will meaningfully improve in the following
30-day window?

**Decision this supports:** A content strategist has limited time and can't manually
review every page. This ranks pages by their likelihood of CTR growth, so review time
goes to the pages most worth a second look — rather than defaulting to "just fix the
pages already ranking well," which turns out to be a weak intuition (see Results).

**Who acts on it:** A content editor or SEO strategist doing weekly/monthly refresh
planning.

**Cost of a wrong call:** Reviewing a page that wasn't actually going to improve wastes
editorial time — a moderate cost, not a dangerous one. This is why the work is framed as
decision-support (a ranked shortlist for a human to check), not an automated action.
"""

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [ ]:
"""
**Release:** `FlyRank/internship-warehouse` (Hugging Face), build
`flyrank_pseudonymized_warehouse_release_v20260703`.

**Tables used:**
- `fact_content_daily_performance` — daily search performance, used to build two
  30-day windows (a "prev30" window and a "last30" window) per content item.
- `dim_content` — content metadata (word count, content type, search intent,
  competition level) joined in as features.

**Date window:** The most recent 60 days of the daily fact table (relative to its max
date), split into two consecutive 30-day halves.

**Rows:** 107,324 content items had at least 50 impressions in *both* windows — this
volume floor was applied before anything else, to stop tiny-denominator pages from
producing meaningless CTR swings.

**Excluded on purpose:**
- `content_updated_date`, `last_optimized_date`, `optimization_eligible_date` — these
  weren't checked for falling safely before the feature window, so including them risked
  leaking information about edits made during the period we're trying to predict.
- `fact_content_query_90d` — its 90-day window overlaps the most recent months of the
  snapshot, which would overlap our label window. Left out entirely rather than risk it.
- No raw URLs, client names, domains, or keyword text appear anywhere — only hashed
  `client_hash_id` / `content_hash_id`, used solely for grouping and joining.
  """

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [ ]:
"""
**Assumptions:** A page's CTR behavior in the next 30 days is at least partly predictable
from its own recent performance and static content properties — not purely random noise.

**Label definition:** `is_ctr_growing_label = 1` when CTR in the "last30" window is at
least 0.3 percentage points higher than CTR in the "prev30" window, else `0`.

We deliberately used an **absolute** point-gain threshold rather than a relative one
(e.g. "grew by 20%"). An earlier version of this label used a relative threshold, and it
produced a suspiciously perfect Precision@50 of 0.98 — investigating showed the model had
just learned "if starting CTR is near zero, predict growth," since a tiny absolute click
gain looks huge in percentage terms when the starting point is near zero. Switching to an
absolute threshold removed that shortcut and brought results down to an honest range.

**Features (never include the label's own ingredients):**
- Numeric: `imp_prev30`, `clk_prev30`, `ctr_prev30`, `avg_position_prev30`, `word_count`,
  `char_count`, `search_volume`, `competition`, `cpc`, `backlinks`
- Categorical: `content_type`, `main_intent`, `competition_level`

**Excluded as leakage:** `imp_last30`, `clk_last30`, `ctr_last30` — these are literally
what the label is computed from.

**Baseline:** A transparent rule scoring pages by (a) how close to page-1 position they
already are, (b) how far their current CTR sits below this dataset's median CTR, and
(c) how much traffic they have. The "weak CTR" cutoff was set to this dataset's own
median `ctr_prev30`, not an arbitrary number — an earlier version borrowed a threshold
from a different dataset's scale and produced a meaningless baseline.

**Validation design:** `GroupShuffleSplit` grouped by `client_hash_id`, 80/20 train/test.
This guarantees no client's pages appear in both train and test — a random row-level
split would let the model partly memorize per-client patterns and overstate its real
performance.

**Leakage checks:** Confirmed no leaky column appears in the feature list via an
assertion that halts the notebook if one sneaks in; feature importances checked for any
single feature suspiciously dominating the model.
"""

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [ ]:
"""
Precision@20:  baseline 0.000   |   random forest 0.350
Precision@50:  baseline 0.000   |   random forest 0.320
Base rate in test set: 0.225
"""

## 5. Limitations

*What this work cannot claim.*

In [ ]:

"""
- **Observed, not causal.** This model finds pages *associated with* future CTR growth.
  It does not identify what to change on a page, and it cannot claim that any edit
  *causes* growth — that would need a controlled experiment (e.g. before/after a real
  edit, compared to a held-out control group).
- **One snapshot, one pair of windows.** The label covers a single 60-day slice of a
  17-month history. Whether this pattern holds in other periods (different seasons,
  different search-algorithm states) is untested here.
- **Threshold choices are ours, not the data's.** The 0.3-percentage-point growth
  threshold and the 50-impression volume floor were chosen deliberately, but they are
  choices — a different threshold could describe a different rate of "success."
- **Content metadata may be incomplete.** Some `dim_content` fields (word count,
  backlinks) are missing for a meaningful share of rows and were filled with 0 rather
  than dropped, which could understate their real importance.
- **Decision-support, not automation.** The ranked output is meant to prioritize human
  review — not to trigger any automatic action.
"""

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [8]:
"""
        avg_position_prev30  ctr_prev30  ctr_last30  imp_prev30  \
60886             5.754623    0.000187    0.000136    230512.0
41526             2.543075    0.000243    0.002000    213610.0
14170            10.091173    0.000011    0.000590    178306.0
11017            14.020197    0.000162    0.000522    191252.0
88867             7.787938    0.000227    0.000187    140919.0
42867             3.065436    0.001254    0.001364    349963.0
11153             7.532791    0.000036    0.000448    112236.0
64953            10.591592    0.000242    0.000436    136464.0
87027             7.710370    0.000127    0.000191    117862.0
96200             2.164448    0.000550    0.000317    129096.0
"""

'\n        avg_position_prev30  ctr_prev30  ctr_last30  imp_prev30  60886             5.754623    0.000187    0.000136    230512.0   \n41526             2.543075    0.000243    0.002000    213610.0   \n14170            10.091173    0.000011    0.000590    178306.0   \n11017            14.020197    0.000162    0.000522    191252.0   \n88867             7.787938    0.000227    0.000187    140919.0   \n42867             3.065436    0.001254    0.001364    349963.0   \n11153             7.532791    0.000036    0.000448    112236.0   \n64953            10.591592    0.000242    0.000436    136464.0   \n87027             7.710370    0.000127    0.000191    117862.0   \n96200             2.164448    0.000550    0.000317    129096.0   \n'

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [7]:
"""
- `work/outputs/charts/precision_comparison.svg` — baseline vs model at Precision@50
- `work/outputs/charts/feature_importance.svg` — what the model actually leans on
- `work/outputs/charts/label_distribution.svg` — how common CTR growth actually is
  in this data (the base rate, visually)
- `work/outputs/ctr_growth_queue.csv` — the full ranked recommendation output

These get copied into `docs/charts/` when the GitHub Pages site is built.
"""
import os
os.makedirs("work/outputs/charts", exist_ok=True)

def simple_svg_bar_chart(title, labels, values, path, color="#4FB8CC"):
    width, height = 960, 420
    margin_left, margin_right, margin_top, margin_bottom = 190, 40, 70, 50
    plot_width = width - margin_left - margin_right
    plot_height = height - margin_top - margin_bottom
    max_value = max(values) if values else 1
    bar_gap = 10
    bar_height = max(14, (plot_height - bar_gap * max(len(values) - 1, 0)) / max(len(values), 1))
    lines = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{width/2}" y="34" text-anchor="middle" font-family="Arial" font-size="22" fill="#16232a">{title}</text>',
    ]
    for i, (label, value) in enumerate(zip(labels, values)):
        y = margin_top + i * (bar_height + bar_gap)
        bar_width = (value / max_value) * plot_width if max_value else 0
        lines.append(f'<text x="{margin_left-12}" y="{y+bar_height*0.65:.1f}" text-anchor="end" font-family="Arial" font-size="13" fill="#27343b">{label}</text>')
        lines.append(f'<rect x="{margin_left}" y="{y:.1f}" width="{bar_width:.1f}" height="{bar_height:.1f}" fill="{color}" rx="4"/>')
        lines.append(f'<text x="{margin_left+bar_width+8:.1f}" y="{y+bar_height*0.65:.1f}" font-family="Arial" font-size="13" fill="#27343b">{value:.3f}</text>')
    lines.append("</svg>")
    with open(path, "w") as f:
        f.write("\n".join(lines))

# Chart 1: Precision@50 comparison
simple_svg_bar_chart(
    "Precision@50: baseline vs random forest",
    ["baseline_rule", "random_forest"],
    [precision_at_k(y_test, baseline_test_scores, 50), precision_at_k(y_test, probabilities, 50)],
    "work/outputs/charts/precision_comparison.svg",
)

# Chart 2: Feature importances
top_features = importances.sort_values(ascending=False).head(10)
simple_svg_bar_chart(
    "Top model features",
    top_features.index.tolist(),
    top_features.values.tolist(),
    "work/outputs/charts/feature_importance.svg",
    color="#86E3D0",
)

# Chart 3: Label distribution (growing vs not)
label_counts = data["is_ctr_growing_label"].value_counts().sort_index()
simple_svg_bar_chart(
    "CTR growth label distribution",
    ["not growing (0)", "growing (1)"],
    label_counts.values.tolist(),
    "work/outputs/charts/label_distribution.svg",
)

print("Charts written to work/outputs/charts/")

Charts written to work/outputs/charts/


In [ ]:
"""
I built and validated a random forest model that
predicts which web pages are likely to see meaningful CTR growth in the next 30 days,
using client-grouped validation on ~107,000 real (anonymized) content items from a
production search-analytics warehouse. Along the way I caught and fixed a label-leakage
artifact that was inflating my initial results, bringing the model to a defensible
[FILL IN]x precision lift over a hand-written baseline. The output is a ranked,
reason-coded recommendation queue designed for a content strategist's weekly review
workflow.
"""

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
